In [ ]:
# ==============================================================================
# CELL 1: The Forge (v1.2 - Hotfix for Logging)
# ==============================================================================
# Objective: Download the entire street network for Mexico City, using the
# correct syntax for enabling verbose progress logging.

# --- 0. INSTALL DEPENDENCIES ---
!pip install osmnx -q

import osmnx as ox
from google.colab import drive
import os

print("--- The Forge: Pre-computation Engine Initializing ---")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')
CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
GRAPH_FILE_PATH = os.path.join(CANONICAL_PATH, 'cdmx_street_network.graphml')
os.makedirs(CANONICAL_PATH, exist_ok=True)
PLACE_QUERY = "Mexico City, Mexico"

# --- CRITICAL HOTFIX: ENABLE VERBOSE LOGGING (MODERN SYNTAX) ---
# The old ox.config() is deprecated. The new, correct way is to set attributes on ox.settings.
ox.settings.log_console = True
# ----------------------------------------------------------------

# --- 2. EXECUTION ---
if not os.path.exists(GRAPH_FILE_PATH):
    print(f"\nStep A.1: Local street graph not found. Fetching from OpenStreetMap...")
    print("-> This is a one-time, long-running operation (5-15 minutes).")
    print("-> You will now see a series of detailed status updates from the osmnx library.")

    # This command downloads the entire graph. With logging enabled, it will be verbose.
    G = ox.graph_from_place(PLACE_QUERY, network_type='drive')

    print("\n -> Success. Street network graph has been downloaded into memory.")

    print(f"\nStep A.2: Saving the graph to a persistent file...")
    ox.save_graphml(G, GRAPH_FILE_PATH)
    print(f" -> SUCCESS: Mexico City street network is now saved locally at: {GRAPH_FILE_PATH}")
else:
    print("\nSUCCESS: Local street graph already exists. No action needed.")
    print("You may proceed to Cell 2 (The Sandbox) or Cell 3 (The Production Line).")

print("\n--- PRE-COMPUTATION COMPLETE ---")

--- The Forge: Pre-computation Engine Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Step A.1: Local street graph not found. Fetching from OpenStreetMap...
-> This is a one-time, long-running operation (5-15 minutes).
-> You will now see a series of detailed status updates from the osmnx library.

 -> Success. Street network graph has been downloaded into memory.

Step A.2: Saving the graph to a persistent file...
 -> SUCCESS: Mexico City street network is now saved locally at: /content/drive/My Drive/_Pienza/Assets/Phase_1/cdmx_street_network.graphml

--- PRE-COMPUTATION COMPLETE ---


In [18]:
# ==============================================================================
# CELL 2: The Jittering Engine v2.4 (Definitive Production Run - Updated Paths)
# ==============================================================================
# Objective: The definitive engine that uses the final 'master_2' data source
# and outputs a fully audited CSV file.

# --- 0. IMPORTS, AUTH & CONFIG ---
import pandas as pd
import gspread
import os
import random
import time
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
from tqdm.auto import tqdm
from thefuzz import process
from google.colab import drive, auth
from google.auth import default

drive.mount('/content/drive'); auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)

# --- Configuration (UPDATED AND FINAL) ---
CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
GRAPH_FILE_PATH = os.path.join(CANONICAL_PATH, 'cdmx_street_network.graphml')
GSHEET_NAME = 'opus_1_phase_1_goeocoding_cleaning_master'
WORKSHEET_NAME = 'master_2' # <-- UPDATED
OUTPUT_CSV_FILE = os.path.join(CANONICAL_PATH, 'master_jittered_for_verification_v3.csv') # <-- UPDATED
SCORE_CUTOFF = 50
BUFFER_RADIUS_METERS = 2000

print("--- The Jittering Engine (v2.4 - Final Production) Initializing ---")
print(f"-> Input Sheet: '{WORKSHEET_NAME}' | Output CSV: '{os.path.basename(OUTPUT_CSV_FILE)}'")

# --- 1. LOAD LOCAL GRAPH & DATA ---
print("\nStep 1: Loading pre-computed street network graph...")
G = ox.load_graphml(GRAPH_FILE_PATH)
gdf_streets = ox.graph_to_gdfs(G, nodes=False, edges=True)
gdf_streets['clean_name'] = gdf_streets['name'].apply(lambda x: x[0] if isinstance(x, list) else x)
print(" -> Success. Local street network is loaded.")

print(f"\nStep 2: Ingesting data from GSheet '{WORKSHEET_NAME}'...")
worksheet = gc.open(GSHEET_NAME).worksheet(WORKSHEET_NAME)
df = pd.DataFrame(worksheet.get_all_records())
for col in ['pickup_lat_v2', 'pickup_lon_v2', 'dropoff_lat_v2', 'dropoff_lon_v2']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- 2. EXECUTE OFFLINE JITTERING WITH AUDITING ---
print("\nStep 3: Executing high-speed, row-by-row remediation with full auditing...")
df['ambiguous_pickup_lat_random'] = None
df['ambiguous_pickup_lon_random'] = None
df['pickup_match_name'] = None
df['pickup_match_score'] = None
df['ambiguous_dropoff_lat_random'] = None
df['ambiguous_dropoff_lon_random'] = None
df['dropoff_match_name'] = None
df['dropoff_match_score'] = None

def get_random_point_on_line(line_geometry):
    if not line_geometry or line_geometry.is_empty: return None
    if line_geometry.geom_type == 'MultiLineString':
        line_lengths = [line.length for line in line_geometry.geoms]
        if not line_lengths or sum(line_lengths) == 0: return None
        chosen_line = random.choices(line_geometry.geoms, weights=line_lengths, k=1)[0]
    else: chosen_line = line_geometry
    if chosen_line.length == 0: return None
    return chosen_line.interpolate(random.uniform(0, chosen_line.length))

pbar = tqdm(df.iterrows(), total=len(df))
for index, row in pbar:
    def process_ambiguous_point(lat, lon, address_str):
        if pd.isna(lat) or pd.isna(lon) or not address_str: return None, None, None, None
        parsed_address = address_str.split(',')[0].strip()
        center_point = Point(lon, lat)
        buffer_poly = center_point.buffer(BUFFER_RADIUS_METERS / 111320)
        local_streets_gdf = gdf_streets[gdf_streets.intersects(buffer_poly)]
        if local_streets_gdf.empty: return None, None, "No_Local_Streets", 0
        available_names = local_streets_gdf['clean_name'].dropna().unique().tolist()
        if not available_names: return None, None, "No_Local_Names", 0
        best_match, score = process.extractOne(parsed_address, available_names)
        if score >= SCORE_CUTOFF:
            target_street_geom = local_streets_gdf[local_streets_gdf['clean_name'] == best_match].geometry.union_all()
            point = get_random_point_on_line(target_street_geom)
            if point: return point.y, point.x, best_match, score
        return None, None, best_match, score

    if row.get('pickup_ambiguity') == 'ambiguous':
        # --- UPDATED: Using 'pickup_address_v3' as the string input ---
        p_lat, p_lon, p_name, p_score = process_ambiguous_point(row['pickup_lat_v2'], row['pickup_lon_v2'], row['pickup_address_v3'])
        df.at[index, 'ambiguous_pickup_lat_random'] = p_lat
        df.at[index, 'ambiguous_pickup_lon_random'] = p_lon
        df.at[index, 'pickup_match_name'] = p_name
        df.at[index, 'pickup_match_score'] = p_score

    if row.get('dropoff_ambiguity') == 'ambiguous':
        # --- Dropoff remains on 'dropoff_address_v2' as per instructions ---
        d_lat, d_lon, d_name, d_score = process_ambiguous_point(row['dropoff_lat_v2'], row['dropoff_lon_v2'], row['dropoff_address_v2'])
        df.at[index, 'ambiguous_dropoff_lat_random'] = d_lat
        df.at[index, 'ambiguous_dropoff_lon_random'] = d_lon
        df.at[index, 'dropoff_match_name'] = d_name
        df.at[index, 'dropoff_match_score'] = d_score

print(" -> Offline jittering and auditing complete.")

# --- 3. EXPORT THE FINAL ARTIFACT TO CSV ---
print(f"\nStep 4: Exporting the audited results to a CSV file...")
df_for_upload = df.fillna('')
df_for_upload.to_csv(OUTPUT_CSV_FILE, index=False)
print(f" -> SUCCESS: The fully audited intermediate artifact has been saved to:")
print(f" -> {OUTPUT_CSV_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- The Jittering Engine (v2.4 - Final Production) Initializing ---
-> Input Sheet: 'master_2' | Output CSV: 'master_jittered_for_verification_v3.csv'

Step 1: Loading pre-computed street network graph...
 -> Success. Local street network is loaded.

Step 2: Ingesting data from GSheet 'master_2'...

Step 3: Executing high-speed, row-by-row remediation with full auditing...


  0%|          | 0/4766 [00:00<?, ?it/s]

 -> Offline jittering and auditing complete.

Step 4: Exporting the audited results to a CSV file...
 -> SUCCESS: The fully audited intermediate artifact has been saved to:
 -> /content/drive/My Drive/_Pienza/Assets/Phase_1/master_jittered_for_verification_v3.csv
